In [34]:
!pip install opencv-python matplotlib

import cv2
import numpy as np
import matplotlib.pyplot as plt


In [35]:
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    img = cv2.imread(filename)


Saving img-2.jpeg to img-2.jpeg


In [36]:
lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
L, A, B = cv2.split(lab)

clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
L2 = clahe.apply(L)

lab_corrected = cv2.merge((L2, A, B))
color_corrected = cv2.cvtColor(lab_corrected, cv2.COLOR_LAB2BGR)


In [37]:
def simple_white_balance(img):
    result = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    avg_a = np.average(result[:,:,1])
    avg_b = np.average(result[:,:,2])
    result[:,:,1] = result[:,:,1] - ((avg_a - 128) * 0.9)
    result[:,:,2] = result[:,:,2] - ((avg_b - 128) * 0.9)
    return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)

wb = simple_white_balance(color_corrected)


In [38]:
bgr = wb.copy()
b = bgr[:,:,2]
b = cv2.equalizeHist(b)
bgr[:,:,2] = b
deseptoned = bgr


In [39]:
def contrast_stretch(img):
    p2, p98 = np.percentile(img, (2, 98))
    return cv2.normalize(img, None, p2, p98, cv2.NORM_MINMAX)

stretched = contrast_stretch(deseptoned)


In [40]:
blur = cv2.GaussianBlur(stretched, (5,5), 3.0)
sharp = cv2.addWeighted(stretched, 1.4, blur, -0.4, 0)


In [41]:
titles = ["Original", "Color Corrected", "White Balance", "Desepia", "Final Restored"]
imgs   = [img, color_corrected, wb, deseptoned, sharp]

plt.figure(figsize=(18,10))
for i in range(5):
    plt.subplot(2,3,i+1)
    plt.imshow(cv2.cvtColor(imgs[i], cv2.COLOR_BGR2RGB))
    plt.title(titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()


Output hidden; open in https://colab.research.google.com to view.

In [43]:
cv2.imwrite("RESTORE_FULL_COLOR.jpg", sharp)
print("Hasil akhir disimpan sebagai: RESTORE_FULL_COLOR.jpg")


Hasil akhir disimpan sebagai: RESTORE_FULL_COLOR.jpg
